# Superstore - Retail

In [1]:
import pandas as pd 
import numpy as np 

In [2]:
data = pd.read_csv("../data/raw/superstore-tableau.csv")

In [3]:
data.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,08-11-2016,11-11-2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,08-11-2016,11-11-2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,12-06-2016,16-06-2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,11-10-2015,18-10-2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,11-10-2015,18-10-2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [4]:
data.columns

Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit'],
      dtype='str')

In [5]:
# Creating unique data tables

customer = data[["Customer ID", "Customer Name", "Segment", "Country", "State", "City", "Postal Code", "Region"]].drop_duplicates()
product = data[["Product ID", "Category", "Sub-Category", "Product Name"]].drop_duplicates()
orders = data[["Customer ID", "Order ID", "Order Date", "Ship Date", "Ship Mode", "Product ID", "Sales", "Quantity", "Discount", "Profit"]].drop_duplicates()
orders['Order Date'] = pd.to_datetime(orders['Order Date'], format = "mixed")
orders['Ship Date'] = pd.to_datetime(orders['Ship Date'], format = "mixed")
orders["days_till_shipped"] = (orders["Ship Date"] - orders["Order Date"]).dt.days

dates = pd.DataFrame(orders['Order Date'].unique()).sort_values(by=0).reset_index(drop=True)
dates.columns = ['date']
dates['month'] = dates['date'].dt.month_name()
dates['month_num'] = dates['date'].dt.month
dates['year'] = dates['date'].dt.year



In [6]:
# Let's check total  orders
print("Total orders:", len(orders['Order ID'].unique()))

# Let's check total  sales
print("Total sales:", round(orders['Sales'].sum(), 2))

# Let's check total  customers
print("Total customers:", len(orders['Customer ID'].unique()))


Total orders: 5009
Total sales: 2296919.49
Total customers: 793


In [7]:
# Total Sales Year-on-Year
sales_yoy = pd.DataFrame(orders.merge(dates[['date','year']], left_on="Order Date", right_on = "date").groupby(['year'])['Sales'].sum())
x1 = sales_yoy.Sales.values[0]
cum_sales = []
for x in sales_yoy.Sales.values:
    if(x==x1):
        cum_sales.append(0)
    else:
        cum_sales.append(round((x-x1)/x,2))
        x1 = x
sales_yoy['%_change_in_sales'] = cum_sales
sales_yoy

,Sales,%_change_in_sales
year,,
2014,483966.1261,0.00
2015,470532.5090,-0.03
2016,609205.5980,0.23
2017,733215.2552,0.17


In [8]:
# Monthly sales across years 2014-2017
sales_2014 = round(pd.DataFrame(orders.merge(dates[dates['year'] == 2014][['date','month_num']], 
                                      left_on="Order Date", 
                                     right_on = "date").groupby(['month_num'])['Sales'].sum()).sort_values('month_num'),2)
sales_2014.columns = ["Sales_2014"]

sales_2015 = round(pd.DataFrame(orders.merge(dates[dates['year'] == 2015][['date','month_num']], 
                                      left_on="Order Date", 
                                     right_on = "date").groupby(['month_num'])['Sales'].sum()).sort_values('month_num'),2)
sales_2015.columns = ["Sales_2015"]

sales_2016 = round(pd.DataFrame(orders.merge(dates[dates['year'] == 2016][['date','month_num']], 
                                      left_on="Order Date", 
                                     right_on = "date").groupby(['month_num'])['Sales'].sum()).sort_values('month_num'),2)
sales_2016.columns = ["Sales_2016"]
sales_2017 = round(pd.DataFrame(orders.merge(dates[dates['year'] == 2017][['date','month_num']], 
                                      left_on="Order Date", 
                                     right_on = "date").groupby(['month_num'])['Sales'].sum()).sort_values('month_num'),2)
sales_2017.columns = ["Sales_2017"]
monthly_sales_all = sales_2014.join([sales_2015, sales_2016, sales_2017])
monthly_sales_all = monthly_sales_all.merge(dates[['month_num',"month"]].drop_duplicates(), "left", on ="month_num")[["month","Sales_2014","Sales_2015","Sales_2016","Sales_2017"]]
monthly_sales_all

,month,Sales_2014,Sales_2015,Sales_2016,Sales_2017
0,January,28953.71,29347.39,38048.18,64734.31
1,February,12743.11,20728.35,49238.41,50011.49
2,March,54801.91,40876.61,49612.04,74774.08
3,April,24428.64,38056.97,45192.28,39072.00
4,May,29639.83,30933.71,64964.32,40882.45
5,June,29287.03,28862.20,38991.94,47742.33
6,July,35341.25,28730.38,42773.40,54382.09
7,August,37854.55,50094.53,46339.99,75675.30
8,September,66110.22,66729.33,41985.14,74164.61
9,October,34561.95,32025.08,52268.15,65501.16


In [9]:
# % change in monthly sales across years 2015-2017
# % of change in sales
monthly_sales_all['%_change_from_2014'] = round((monthly_sales_all['Sales_2015'] - monthly_sales_all['Sales_2014'])/
                                               monthly_sales_all['Sales_2015'],2)
monthly_sales_all['%_change_from_2015'] = round((monthly_sales_all['Sales_2016'] - monthly_sales_all['Sales_2015'])/
                                               monthly_sales_all['Sales_2016'],2)
monthly_sales_all['%_change_from_2016'] = round((monthly_sales_all['Sales_2017'] - monthly_sales_all['Sales_2016'])/
                                               monthly_sales_all['Sales_2017'],2)
monthly_sales_all = monthly_sales_all[["month","Sales_2014","Sales_2015","%_change_from_2014","Sales_2016","%_change_from_2015","Sales_2017","%_change_from_2016"]]
monthly_sales_all

,month,Sales_2014,Sales_2015,%_change_from_2014,Sales_2016,%_change_from_2015,Sales_2017,%_change_from_2016
0,January,28953.71,29347.39,0.01,38048.18,0.23,64734.31,0.41
1,February,12743.11,20728.35,0.39,49238.41,0.58,50011.49,0.02
2,March,54801.91,40876.61,-0.34,49612.04,0.18,74774.08,0.34
3,April,24428.64,38056.97,0.36,45192.28,0.16,39072.00,-0.16
4,May,29639.83,30933.71,0.04,64964.32,0.52,40882.45,-0.59
5,June,29287.03,28862.20,-0.01,38991.94,0.26,47742.33,0.18
6,July,35341.25,28730.38,-0.23,42773.40,0.33,54382.09,0.21
7,August,37854.55,50094.53,0.24,46339.99,-0.08,75675.30,0.39
8,September,66110.22,66729.33,0.01,41985.14,-0.59,74164.61,0.43
9,October,34561.95,32025.08,-0.08,52268.15,0.39,65501.16,0.20


In [10]:
round(monthly_sales_all["%_change_from_2014"].mean(),4), monthly_sales_all["%_change_from_2015"].mean(), round(monthly_sales_all["%_change_from_2016"].mean(),4)

(np.float64(-0.0092), np.float64(0.2075), np.float64(0.1167))

In [11]:
round(monthly_sales_all["Sales_2014"].sum(),4), round(monthly_sales_all["Sales_2015"].sum(),4), round(monthly_sales_all["Sales_2016"].sum(),4), round(monthly_sales_all["Sales_2017"].sum(),4)

(np.float64(483966.13),
 np.float64(470532.52),
 np.float64(609205.6),
 np.float64(733215.26))

The year 2016 observed a 25% of increase in sales in comparison to the previous year. However, 2017 witnessed only 20% of increase in sales in comparison to the previous year, indicating a 5% of drop in sales if consistency was witnessed

In [ ]:
orders.head()

In [12]:
# creating aggregated view of orders
columns = ['Order ID', 'Order Date', 'year', 'month', "month_num", 'Ship Mode', 'days_till_shipped', 'Sales',
       'Quantity', 'Product ID' ]
orders_agg = orders.groupby(["Order ID", 
                             "Order Date", 
                             "Ship Mode",
                             "days_till_shipped"])[
                                 ["Sales",
                                  "Quantity"]
                                 ].sum().reset_index().merge(
                                     orders.groupby("Order ID")["Product ID"].count(),
                                      on="Order ID").merge(dates[["date",
                                                        "year","month","month_num"]], 
                                                        left_on="Order Date", 
                                                        right_on = "date")[columns]
orders_agg = orders_agg.sort_values('Order Date')
orders_agg

,Order ID,Order Date,year,month,month_num,Ship Mode,days_till_shipped,Sales,Quantity,Product ID
495,CA-2014-140795,2014-01-02,2014,January,1,First Class,59,468.900,6,1
62,CA-2014-104269,2014-01-03,2014,January,1,Second Class,151,457.568,2,1
786,CA-2014-168312,2014-01-03,2014,January,1,Standard Class,181,513.861,6,2
176,CA-2014-113880,2014-01-03,2014,January,1,Standard Class,120,651.588,9,2
4294,US-2014-143707,2014-01-03,2014,January,1,Standard Class,120,5.940,3,1
...,...,...,...,...,...,...,...,...,...,...
4965,US-2017-158526,2017-12-29,2017,December,12,Second Class,3,1814.680,14,5
3928,CA-2017-156720,2017-12-30,2017,December,12,Standard Class,61,3.024,3,1
3663,CA-2017-143259,2017-12-30,2017,December,12,Standard Class,61,466.842,14,3
3097,CA-2017-115427,2017-12-30,2017,December,12,Standard Class,61,34.624,4,2


In [13]:
# Total orders shipped for each ship mode through 2014-2017
orders_agg.groupby(["Ship Mode","year"])["Order ID"].count().reset_index().pivot(
    index = "Ship Mode",
    columns=["year"], 
    values = "Order ID")

year,2014,2015,2016,2017
Ship Mode,,,,
First Class,145,143,215,284
Same Day,48,53,74,89
Second Class,190,206,244,324
Standard Class,586,636,782,990


- Standard class ship mode - shipped highest number of orders in all three years, however same day orders shipped the lowest orders all through years

In [14]:
# median days till orders shipped for each ship mode through 2014-2017
round(orders_agg.groupby(["Ship Mode","year"])["days_till_shipped"].median(),2).reset_index().pivot(
    index = "Ship Mode",
    columns=["year"], 
    values = "days_till_shipped")

year,2014,2015,2016,2017
Ship Mode,,,,
First Class,3.0,3.0,3.0,3.0
Same Day,0.0,0.0,0.0,0.0
Second Class,3.0,3.0,3.0,4.0
Standard Class,5.0,5.0,5.0,5.0


- Throughout the years, each ship modes showed a consistent delivery times approximately

In [15]:
# Avg days till orders shipped for each ship mode through 2014-2017
round(orders_agg.groupby(["Ship Mode","year"])["days_till_shipped"].mean()).reset_index().pivot(
    index = "Ship Mode",
    columns=["year"], 
    values = "days_till_shipped")

year,2014,2015,2016,2017
Ship Mode,,,,
First Class,13.0,9.0,10.0,2.0
Same Day,0.0,1.0,-1.0,1.0
Second Class,2.0,4.0,2.0,5.0
Standard Class,12.0,12.0,5.0,14.0


In [16]:
# Avg days till orders shipped for each month through 2014-2017
round(orders_agg.groupby(["month_num","month","year"])["days_till_shipped"].median()).reset_index().pivot(
    index = ["month_num","month"],
    columns=["year"], 
    values = "days_till_shipped").reset_index()

year,month_num,month,2014,2015,2016,2017
0,1,January,76.0,90.0,60.0,45.0
1,2,February,120.0,59.0,60.0,89.0
2,3,March,5.0,6.0,6.0,19.0
3,4,April,5.0,5.0,6.0,5.0
4,5,May,6.0,7.0,6.0,4.0
5,6,June,6.0,5.0,4.0,4.0
6,7,July,6.0,5.0,5.0,5.0
7,8,August,5.0,4.0,4.0,4.0
8,9,September,4.0,4.0,4.0,4.0
9,10,October,2.0,2.0,4.0,2.0


- January and february consistently observed highest time till orders were shipped

In [17]:
# Orders for each month through 2014-2017
round(orders_agg.groupby(["month_num","month","year"])["Order ID"].count()).reset_index().pivot(
    index = ["month_num","month"],
    columns=["year"], 
    values = "Order ID").reset_index()

year,month_num,month,2014,2015,2016,2017
0,1,January,58,47,77,102
1,2,February,47,52,62,107
2,3,March,80,87,96,156
3,4,April,60,75,94,134
4,5,May,77,92,123,112
5,6,June,71,81,91,127
6,7,July,70,69,105,122
7,8,August,74,79,117,122
8,9,September,113,127,139,182
9,10,October,63,81,122,145


- Eventhough January and february had highest shipment delays, the total orders processed were still less in comparison to other months, let's check for the quantities and products as well

In [18]:
# Quantities for each month through 2014-2017
round(orders_agg.groupby(["month_num","month","year"])["Quantity"].sum()).reset_index().pivot(
    index = ["month_num","month"],
    columns=["year"], 
    values = "Quantity").reset_index()

year,month_num,month,2014,2015,2016,2017
0,1,January,508,336,563,923
1,2,February,319,395,485,895
2,3,March,611,570,734,1138
3,4,April,468,552,663,803
4,5,May,551,697,1035,892
5,6,June,497,539,696,879
6,7,July,590,526,771,898
7,8,August,604,706,922,981
8,9,September,903,1041,868,1382
9,10,October,555,593,926,991


In [19]:
# Products for each month through 2014-2017
round(orders_agg.groupby(["month_num","month","year"])["Product ID"].sum()).reset_index().pivot(
    index = ["month_num","month"],
    columns=["year"], 
    values = "Product ID").reset_index()

year,month_num,month,2014,2015,2016,2017
0,1,January,131,86,154,226
1,2,February,86,102,126,234
2,3,March,168,154,193,304
3,4,April,120,159,188,229
4,5,May,148,177,260,241
5,6,June,137,152,189,229
6,7,July,156,132,208,244
7,8,August,150,178,237,251
8,9,September,239,278,236,385
9,10,October,145,155,236,272


In [20]:
# Total sales made from each ship mode through 2014-2017
round(orders_agg.groupby(["Ship Mode","year"])["Sales"].sum(),2).reset_index().pivot(
    index = "Ship Mode",
    columns=["year"], 
    values = "Sales")

year,2014,2015,2016,2017
Ship Mode,,,,
First Class,59769.26,69259.44,82265.34,140134.38
Same Day,17470.13,27611.49,34505.76,48775.74
Second Class,101386.78,89102.73,120002.44,148701.62
Standard Class,305339.95,284558.85,372432.05,395603.52


In [41]:
# Shipment time based on category
round(orders_agg.merge(data[["Order ID","Category"]].drop_duplicates(),
                 "left").groupby(["Category","year"])["days_till_shipped"].mean(), 2).reset_index().pivot(
    index = "Category",
    columns=["year"], 
    values = "days_till_shipped")

year,2014,2015,2016,2017
Category,,,,
Furniture,4.52,14.36,4.00,8.42
Office Supplies,11.61,9.92,7.04,8.67
Technology,8.43,6.34,4.42,11.47


In [44]:
# Shipment based on category for each month through 2014-2017
round(orders_agg.merge(data[["Order ID","Category"]].drop_duplicates(),
                 "left").groupby(["Category","month_num","month","year"])["days_till_shipped"].mean()).reset_index().pivot(
    index = ["Category","month_num","month"],
    columns=["year"], 
    values = "days_till_shipped").reset_index()

year,Category,month_num,month,2014,2015,2016,2017
0,Furniture,1,January,58.0,86.0,64.0,73.0
1,Furniture,2,February,97.0,89.0,77.0,78.0
2,Furniture,3,March,33.0,66.0,56.0,71.0
3,Furniture,4,April,38.0,46.0,43.0,30.0
4,Furniture,5,May,44.0,72.0,31.0,7.0
5,Furniture,6,June,48.0,29.0,18.0,22.0
6,Furniture,7,July,37.0,32.0,31.0,12.0
7,Furniture,8,August,29.0,10.0,-9.0,13.0
8,Furniture,9,September,-44.0,-1.0,-37.0,-9.0
9,Furniture,10,October,-40.0,-23.0,-38.0,-44.0
